# Explore preprocessing in stripes project

In [ ]:
%load_ext autoreload
%autoreload 2
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
import subprocess
import scipy.io as sio
import os
import glob
opj = os.path.join

from cvl_utils.preproc_func import run_cmd, set_project
from dpu_mini.fs_tools import load_benson14_info
set_project('stripe')
docker = 'ndock_fsl_freesurfer:latest'
from achstripes.utils import *


In [ ]:
subject = 'sub-0417_c'
ses_list = ['ses-1', 'ses-2']
from cvl_utils.coreg import (
    make_bref_main,
    convert_fs_t1,
    run_bbregister,
    register_sbref_to_main,
    run_mcflirt,
    concat_transforms,
    apply_xfm4d,
    project_to_surface,
)
sub_src_dir = opj(os.environ['BIDS_DIR'], subject)
# subject_input_dir = opj(os.environ['BIDS_DIR'], subject, 'ses-1','func')
src_seq_files = {}

seq_list = ['1mmTR2400', '1mmTR3000', 'pt9mm', '1mmFatSat']
for ses in ses_list:
    subject_input_dir = opj(sub_src_dir, ses, 'func')
    src_seq_files[ses] = {}
    for seq in seq_list:
        slist =  glob.glob(opj(subject_input_dir, f'*{seq}*.nii'))
        if len(slist) > 0:
            src_seq_files[ses][seq] = [s for s in slist if 'whole' not in s]
src_seq_files['ses-2']['1mmFatSat']



#### for og res

In [ ]:
tr_to_chop = 1
full_run_time = {'col' :270, 'mot' : 240} 
ogres_dir = opj(os.environ['BIDS_DIR'], 'derivatives', 'multi_chop', subject)
og_nseq_files = {}
for ses in ses_list:
    og_nseq_files[ses] = []
    seq_out = opj(ogres_dir, ses)
    for seq in seq_list:

        if seq in src_seq_files[ses].keys():
            print(f"Session {ses}, sequence {seq}:")
            for trun in src_seq_files[ses][seq]:
                # if 'run-16' in trun:
                #     print(f"Skipping {trun} because it is run-16")
                #     continue
                img = nib.load(trun)
                TR = img.header.get_zooms()[-1]
                if 'col' in trun:
                    max_vols = int(full_run_time['col'] / TR)
                elif 'mot' in trun:
                    max_vols = int(full_run_time['mot'] / TR)

                bname = os.path.basename(trun).replace('.nii', f'chop{tr_to_chop}-{max_vols}.nii')            
                print(f"Run {bname}: TR={TR:.2f}s, max_vols={max_vols}, ")
                tfile = opj(seq_out, bname)
                og_nseq_files[ses].append(tfile)
                if not os.path.exists(seq_out):
                    os.makedirs(seq_out)
                if True: #not os.path.exists(tfile):
                    data = img.get_fdata()
                    bloop
                    data = data[..., tr_to_chop:max_vols]
                    new_img = nib.Nifti1Image(
                        data, affine=img.affine, header=img.header
                    )
                    nib.save(new_img, tfile)


In [ ]:
data.shape

In [ ]:
# Make bref main and align it... 
fsl_aligned = ogres_dir = opj(os.environ['BIDS_DIR'], 'derivatives', 'multi_align', subject)
if not os.path.exists(fsl_aligned):
    os.makedirs(fsl_aligned)
# make a bref main and coregister everyone to this
bref_main = opj(fsl_aligned, 'sub-0417_c_BREF_MAIN.nii.gz')

tbref_main_src = og_nseq_files['ses-1'][3] # pick one of the 1mmTR3000
with open(opj(fsl_aligned, 'notes.txt'), 'w') as file:
    file.write('BREF_MAIN taken as volume 0 from:')
    file.write(f'{tbref_main_src}')

if True: #not os.path.exists(bref_main):
    img = nib.load(tbref_main_src)
    data = img.get_fdata(dtype=np.float32)
    vol  = data[..., 0] if data.ndim == 4 else data
    out  = nib.Nifti1Image(vol, img.affine, ) #img.header)
    out.set_data_dtype(np.float32)
    nib.save(out, bref_main)
    env = dict(
        BIDS_DIR=os.environ['BIDS_DIR'],
        SUBJECTS_DIR=os.environ['SUBJECTS_DIR'],
    )
    run_cmd(
        ['fslreorient2std', bref_main, bref_main],
        env_vars=env,
        work_dir=os.environ['BIDS_DIR'],
        docker_image=os.environ['FSL_FREESURFER_IMAGE'],
    ) 


In [ ]:
%%bash

# $PYPACKAGE_MANAGER run -n preproc \
#     python $PIPELINE_DIR/functional/s02_coreg_bref_only.py \
#     --bids-dir      $BIDS_DIR \
#     --input-file    multi_chop \
#     --output-file   multi_align \
#     --sub           0417_c \
#     --include-patterns 'ses-*/*_task*.nii' \
#     --subjects-dir  $SUBJECTS_DIR \
#     --docker        $FSL_FREESURFER_IMAGE \
#     --bref-main     sub-0417_c_BREF_MAIN.nii.gz
# #     # --include-patterns 'ses-01/*_task-pRF*bold*.nii.gz' \\
# #     # --bref-main   sub-01_ses-01_task-pRF_run-01_sbref.nii.gz

echo s02_coreg_bref_only_antsSyN.py \
    --bids-dir      $BIDS_DIR \
    --input-file    multi_chop \
    --output-file   multi_align \
    --sub           0417_c \
    --include-patterns 'ses-*/*_task*.nii' \
    --subjects-dir  $SUBJECTS_DIR \
    --docker        $FSL_FREESURFER_IMAGE \
    --bref-main     sub-0417_c_BREF_MAIN.nii.gz


In [ ]:
# Find aligned files 

all_mcf = glob.glob(opj(fsl_aligned, f'ses-*', '*space-*'))
len(all_mcf)

# Glmsingle 

In [ ]:
from achstripes.fslfs_reg import fslfs_reg
fs_dir = os.environ['SUBJECTS_DIR']
soutput_dir = opj(os.environ['BIDS_DIR'], 'derivatives', 'multi_glm', subject)
if not os.path.exists(soutput_dir):
    os.makedirs(soutput_dir)
b14_info = load_benson14_info(subject, fs_dir)
add_ref_anat_and_vol_rois(subject, fs_dir, soutput_dir)
anat_ref = opj(soutput_dir, 'anat_ref.nii.gz')


In [ ]:
reg_dat = opj(fsl_aligned, f'{subject}_desc-sbref2fs_bbr.dat')
def fslfs_func2anat_NEW(func, sub, reg_dat, interp='trilin'):
    run_cmd(
        ['fslreorient2std', func, func],
        env_vars=env,
        work_dir=os.environ['BIDS_DIR'],
        docker_image=os.environ['FSL_FREESURFER_IMAGE'],
    )
    reg_lta = opj(os.path.dirname(reg_dat), 'tmp.lta')
    out = func.replace('.nii.gz', 'al.nii.gz')
    run_cmd(
        [
            'tkregister2',
            '--mov', func,
            '--reg', reg_dat,
            '--targ', opj(env['SUBJECTS_DIR'], sub, 'mri', 'brain.mgz'),
            '--ltaout', reg_lta,
        ],
        env_vars=env,
        work_dir=os.environ['BIDS_DIR'],
        docker_image='local',
    )

    run_cmd(
        [
            'mri_vol2vol',
            '--mov', func,
            '--targ', opj(env['SUBJECTS_DIR'], sub, 'mri', 'brain.mgz'),
            '--lta', reg_lta,
            '--interp', interp,
            '--o', out,
        ],
        env_vars=env,
        work_dir=os.environ['BIDS_DIR'],
        docker_image='local',
    )


# Load DM

In [ ]:
from achstripes.glm import dm_from_path

BIDS_dir = os.environ['BIDS_DIR']
# pt9mm TR=3, 90 timpts (270s)
# 1mmTR3000, TR=3, 90 timepts (270s)
# 1mmTR2400, TR=2.4, 112 timepts (300s)
seq_trs = {
    'pt9mm' : 3, 
    '1mmTR3000' : 3,
    '1mmTR2400' : 2.4,
    '1mmFatSat' : 3.065,
}

task_durs = {
    'bwcol' : 30, 
    'mot' : 16,
}
task_length_secs = {
    'bwcol' : 270, 
    'mot' : 240,
}

dm_dir = opj(BIDS_dir, 'derivatives', 'dm_hacky', subject)



# -> bwcol
dm_pattern = opj(dm_dir, f'*/Run_1', '*/*_Cond.mat')
dm_path = glob.glob(dm_pattern)[0]
tdm_mat = sio.loadmat(dm_path)
dm_conds = [str(i[0]) for i in tdm_mat['names'][0]]

ons_sec_bwcol = {}
for iC,c in enumerate(dm_conds):
    ons_sec_bwcol[c] = tdm_mat['onsets'][0][iC][0] # weird matlab... have to index this way

#
bw_vals = [(v, 'bw', i) for i, v in enumerate(ons_sec_bwcol['bw'])]
col_vals = [(v, 'colour', i) for i, v in enumerate(ons_sec_bwcol['colour'])]
all_vals = sorted(bw_vals + col_vals, key=lambda x: x[0])
bw_order = [None] * len(ons_sec_bwcol['bw'])
col_order = [None] * len(ons_sec_bwcol['colour'])
for rank, (val, source, original_idx) in enumerate(all_vals):
    if source == 'bw':
        bw_order[original_idx] = rank
    else:
        col_order[original_idx] = rank
beta_order_bwcol = {'bw': bw_order, 'col': col_order}


dm_seq_bwcol = {}

for seq in seq_list:
    total_trs = int(task_length_secs['bwcol'] / seq_trs[seq])
    print(f"Sequence {seq}: total TRs={total_trs}")
    dm_seq_bwcol[seq] = np.zeros((total_trs, len(dm_conds)-1))
    for iC,c in enumerate(dm_conds[1::]): # skip the first condition, which is rest
        ons_trs = np.array(ons_sec_bwcol[c]) / seq_trs[seq]
        ons_trs = np.round(ons_trs).astype(int)
        dm_seq_bwcol[seq][ons_trs, iC] = 1



In [ ]:
import pandas as pd

def audit_onset_binning(ons_sec, dm_conds, seq_trs, task_durs=None, task=None, cond_offset=1):
    """
    For every (sequence, condition) pair, compare true onset times (s) to the
    times implied by round(onset/TR) binning: how much jitter does the TR
    grid introduce, and does any onset collide into an already-used TR bin
    (= a trial silently dropped from the design matrix)?
    """
    rows, collisions = [], {}
    for seq, tr in seq_trs.items():
        for c in dm_conds[cond_offset:]:
            true_ons = np.asarray(ons_sec[c], dtype=float)
            tr_idx = np.round(true_ons / tr).astype(int)
            err = true_ons - tr_idx * tr  # signed, seconds
            row = {
                'seq': seq, 'tr': tr, 'cond': c, 'n_events': len(true_ons),
                'mean_err_s': err.mean(), 'mean_abs_err_s': np.abs(err).mean(),
                'max_abs_err_s': np.abs(err).max(),
                'mean_abs_err_%tr': 100 * np.abs(err).mean() / tr,
            }
            if task_durs and task:
                row['mean_abs_err_%dur'] = 100 * np.abs(err).mean() / task_durs[task]
            rows.append(row)

            uniq, counts = np.unique(tr_idx, return_counts=True)
            for b in uniq[counts > 1]:
                collisions[(seq, c)] = collisions.get((seq, c), [])
                collisions[(seq, c)].append((b, true_ons[tr_idx == b].tolist()))

    df = pd.DataFrame(rows).sort_values(['seq', 'cond'])
    return df, collisions

err_df, collisions = audit_onset_binning(ons_sec_bwcol, ['resst', 'bw', 'colour'], seq_trs,
                                          task_durs=task_durs, task='bwcol')
print(err_df.to_string(index=False))
if collisions:
    print("\nCOLLISIONS — a trial is being dropped here:")
    for (seq, c), bins in collisions.items():
        print(f"  {seq}/{c}: {bins}")
else:
    print("\nNo collisions — every event still gets its own TR row.")


err_df, collisions = audit_onset_binning(ons_sec_mot, ['rest', 'mot'], seq_trs,
                                          task_durs=task_durs, task='mot')
print(err_df.to_string(index=False))
if collisions:
    print("\nCOLLISIONS — a trial is being dropped here:")
    for (seq, c), bins in collisions.items():
        print(f"  {seq}/{c}: {bins}")
else:
    print("\nNo collisions — every event still gets its own TR row.")

In [ ]:
task_durs

In [ ]:
ons_sec_bwcol

In [ ]:
ons_sec_mot

In [ ]:
ons_sec_bwcol

In [ ]:
ons_sec_bwcol

In [ ]:
ons_sec_bwcol
ons_sec_mot

In [ ]:

# -> mot 
dm_pattern = opj(dm_dir, f'*/*/Run_1','*/*.mat')
dm_path = glob.glob(dm_pattern)[0]
tdm_mat = sio.loadmat(dm_path)
dm_conds = [str(i[0]) for i in tdm_mat['names'][0]]
ons_sec_mot = {}
ons_sec_mot['mot'] = []

for iC,c in enumerate(dm_conds):
    if 'ori' in c:
        ons_sec_mot['mot'].append(tdm_mat['onsets'][0][iC][0]) # weird matlab... have to index this way
    else:
        ons_sec_mot[c] = tdm_mat['onsets'][0][iC][0]
ons_sec_mot['mot'] = np.sort(np.concatenate(ons_sec_mot['mot']))

beta_order_mot = {'mot': list(range(len(ons_sec_mot['mot'])))}
dm_seq_mot = {}
for seq in seq_list:
    total_trs = int(task_length_secs['mot'] / seq_trs[seq])
    print(f"Sequence {seq}: total TRs={total_trs}")
    dm_seq_mot[seq] = np.zeros((total_trs, 1))
    ons_trs = np.array(ons_sec_mot['mot']) / seq_trs[seq]
    print(ons_trs)
    ons_trs = np.round(ons_trs).astype(int)
    dm_seq_mot[seq][ons_trs, 0] = 1


# combine
dm_seq = {
    'bwcol' : dm_seq_bwcol,
    'mot' : dm_seq_mot,
}
# CHOP FIRST TR
for k1 in dm_seq.keys():
    for k2 in dm_seq[k1].keys(): 
        dm_seq[k1][k2] = dm_seq[k1][k2][1:,:]
# Beta orders 
beta_order = {
    'bwcol' : beta_order_bwcol,
    'mot' : beta_order_mot,
}


In [ ]:
ons_sec['mot']

In [ ]:
import glmsingle
from glmsingle.glmsingle import GLM_single
import scipy
import os
import shutil
opj = os.path.join

opt = dict()
# set important fields for completeness (but these would be enabled by default)
opt['wantlibrary'] = 1
opt['wantglmdenoise'] = 1
opt['wantfracridge'] = 0

# for the purpose of this example we will keep the relevant outputs in memory
# and also save them to the disk
opt['wantfileoutputs'] = [1,1,1,1]
opt['wantmemoryoutputs'] = [0,0,1,0]

# opt['chunklen'] = 500
# running python GLMsingle involves creating a GLM_single object
# and then running the procedure using the .fit() routine
def get_motion_params(chopped_run_file, work_dir):
    out_prefix = opj(work_dir, os.path.basename(chopped_run_file).replace('.nii', '').replace('.gz', '') + '_mcf')
    par_file = out_prefix + '.par'
    # if not os.path.exists(par_file):
    #     run_cmd(
    #         ['mcflirt', '-in', chopped_run_file, '-out', out_prefix, '-plots'],
    #         env_vars=env,
    #         work_dir=os.environ['BIDS_DIR'],
    #         docker_image=os.environ['FSL_FREESURFER_IMAGE'],
    #     )
    return np.loadtxt(par_file)  # (nTRs_full, 6): rot_x, rot_y, rot_z, trans_x, trans_y, trans_z


In [ ]:
# variables that will contain bold time-series and design matrices from each run
glms = {}
results = {}
mot_par_dir = opj(soutput_dir, 'motion_params')
for seq in seq_list:
    # for task in ['bwcol', 'mot']:
    for task in ['bwcol', 'photmot', 'scotmot']:
        task_k = 'mot' if 'mot' in task else task

        trun_files = [i for i in all_mcf if (task in i) and (seq in i)]
        nruns = len(trun_files)
        if nruns == 0:
            continue
        tmot_files = [i.replace(
            '_space-brefmain_desc-moco_bold.nii.gz', 
            '_desc-mcflirt_motion.par'
        ) for i in trun_files]
        # match each aligned run back to its chopped (pre cross-run-alignment) source,
        # same filename-matching logic as everywhere else in the notebook
        run_extra_regressors = []
        for R in range(nruns):
            mp = np.loadtxt(tmot_files[R])

            run_extra_regressors.append(mp)

        run_opt = dict(opt)                              # per-iteration copy, nruns varies by task/seq
        run_opt['extra_regressors'] = run_extra_regressors
        # blorp


        data = []
        design = []
        print(f'for {task} {seq} nruns={nruns}')
        for R in range(nruns):
            data.append(
                nib.load(trun_files[R]).get_fdata()
                )
            design.append(dm_seq[task_k][seq])

        # get total number of blocks - this will be the dimensionality of output betas from GLMsingle
        nblocks = int(np.sum(np.concatenate(design)))

        # get metadata about stimulus duration and TR

        glms[f'{task}-{seq}'] = GLM_single(opt)
        # run GLMsingle
        tglm_path = opj(soutput_dir, f'{task}-{seq}-whrf-motregs')

        if not os.path.exists(tglm_path):                
            results[f'{task}-{seq}'] = glms[f'{task}-{seq}'].fit(
                design,
                data,
                task_durs[task_k],
                seq_trs[seq],
                outputdir=tglm_path
                )
        # else:
        glms_files = ['RUNWISEFIR.npy', 'TYPEB_FITHRF.npy', 'DESIGNINFO.npy', 'TYPEA_ONOFF.npy', 'TYPEC_FITHRF_GLMDENOISE.npy']
        glms_keys = ['runwisefir',      'typeb',            'designinfo',     'typea',            'typec', ]
        results[f'{task}-{seq}'] = {}
        for f, k in zip(glms_files, glms_keys):
            results[f'{task}-{seq}'][k] = np.load(opj(tglm_path, f), allow_pickle=True).item()
        

        from achstripes.glm_qc import build_glmsingle_qc_report, save_glmsingle_asnii
        if not os.path.exists(opj(tglm_path, f'{task}-{seq}_qc.html')):
            build_glmsingle_qc_report(
                opj(tglm_path, f'{task}-{seq}_qc.html'),
                subject=subject, label=f'{task}-{seq}',
                typec=results[f'{task}-{seq}']['typec'],
                runwisefir=results[f'{task}-{seq}']['runwisefir'],
                designinfo=results[f'{task}-{seq}']['designinfo'],
            )
        save_glmsingle_asnii(
            tglm_path, 
            ref_nii=bref_main
        )

In [ ]:
print(bref_main)

In [ ]:
glms_files = ['RUNWISEFIR.npy', 'TYPEB_FITHRF.npy', 'DESIGNINFO.npy', 'TYPEA_ONOFF.npy', 'TYPEC_FITHRF_GLMDENOISE.npy']
glms_keys = ['runwisefir',      'typeb',            'designinfo',     'typea',            'typec', ]
bleep = {}
bleep[f'{task}-{seq}'] = {}
for f, k in zip(glms_files, glms_keys):
    bleep[f'{task}-{seq}'][k] = np.load(opj(tglm_path, f), allow_pickle=True).item()

In [ ]:
bleep[f'{task}-{seq}']['runwisefir']

In [ ]:
plt.plot(run_opt['extra_regressors'][1])

In [ ]:
!pip install plotly

In [ ]:
from achstripes.glm import mean_betas_from_runs
# brainmask = meanvol > 2500


glm_nii_dir = opj(soutput_dir, 'glmsingle-nii', ) 
if not os.path.exists(glm_nii_dir):
    os.makedirs(glm_nii_dir)

betas_full = {}
betas_run = {}
vol_files = {}
for seq in seq_list:
    for task in ['bwcol', 'scotmot', 'photmot']:
        if 'mot' in task:
            task_k = 'mot'
        else:
            task_k = task
        trun_files = all_mcf
        trun_files = [i for i in trun_files if task in i]
        trun_files = [i for i in trun_files if seq in i]
        # rnib_ref = nib.load(brefs[seq])
        tnruns = len(trun_files)
        if tnruns==0:
            continue
        rnib_ref = nib.load(trun_files[0])
        betas_full[f'{task}-{seq}'], betas_run[f'{task}-{seq}'] = mean_betas_from_runs(
            results[f'{task}-{seq}']['typec']['betasmd'],
            tnruns, 
            beta_order[task_k],
        )
        # over all runs 
        if task == 'bwcol':
            tdata_full = betas_full[f'{task}-{seq}']['col'] -  betas_full[f'{task}-{seq}']['bw']
        else:
            tdata_full = betas_full[f'{task}-{seq}']['mot']
        tfile_full = opj(glm_nii_dir, f'{task}-{seq}.nii.gz')
        tfile_full_al = opj(glm_nii_dir, f'{task}-{seq}al.nii.gz')

        timg = nib.Nifti1Image(tdata_full.astype(np.float32), rnib_ref.affine)
        if not os.path.exists(tfile_full):
            nib.save(timg, tfile_full)
            fslfs_func2anat_NEW(tfile_full, subject, reg_dat)
        vol_files[f'{task}-{seq}-all'] = tfile_full_al

        for run in range(tnruns):
            if task == 'bwcol':
                tdata = betas_run[f'{task}-{seq}']['col'][run]-betas_run[f'{task}-{seq}']['bw'][run]
            elif task == 'mot':
                tdata = betas_run[f'{task}-{seq}']['mot'][run]
            timg = nib.Nifti1Image(tdata.astype(np.float32), rnib_ref.affine)
            tfile = opj(glm_nii_dir, f'{task}-{seq}_mv-r{run}.nii.gz')
            tfile_al = opj(glm_nii_dir, f'{task}-{seq}_mv-r{run}al.nii.gz')
            if not os.path.exists(tfile):
                nib.save(timg, tfile)
                fslfs_func2anat_NEW(tfile, subject, reg_dat)

            vol_files[f'{task}-{seq}-r{run}'] = tfile_al


In [ ]:
glm_nii_dir

In [ ]:
import cortex
cortex.database.db.reload_subjects()
# Force the reload

vols = {}
for k in vol_files.keys():
    if 'all' in k:
        vols[k] = nib.load(vol_files[k]).get_fdata().transpose(2,1,0)

maps = {}
for v in vols.keys():
    if v not in ('tsnr', 'mepi'):        
        maps[v] = cortex.Volume(
            vols[v], subject, xfmname='identity',
            vmin=-2, vmax=2,
            cmap='RdBu_r', 
        )
    else:
        maps[v] = cortex.Volume(
            vols[v], subject, xfmname='identity',
            # vmin=-3, vmax=3,
            cmap='jet'
        )

maps['b14ecc']= cortex.Vertex(
    b14_info['ecc'], subject, 
    cmap='jet', vmin=0, vmax=10, 
)
maps['b14pol']= cortex.Vertex(
    b14_info['pol'], subject, 
    cmap='hsv', vmin=0, vmax=180, 
)



In [ ]:
cortex.webshow(
    maps, )

In [ ]:
np.unique(results['bwcol-1mmTR2400']['typec']['HRFindex'])